# Release-Tag Mining — building the snapshot grid (v4)

**What this notebook does.** It selects the study's *observation points*: the GA release snapshots
at which we later ask "what should the team refactor next?". It documents the mining strategy end
to end so the grid is reproducible and every snapshot has a stated reason for being there.

**The strategy in one paragraph.** Take every tag; keep the GA (generally-available) releases and
drop pre-releases. A GA release is a real observation point *because it shipped* — users ran it —
regardless of which branch it was tagged on. Most releases sit on the main line and are used
directly. Some are tagged off the main line, on a short release branch; if that branch was later
merged back, the release's code is already on the main line and we pin the snapshot to that
main-line commit. If it was never merged, the release still counts — we keep it as-is. The only
tags we cannot place are pre-import ones with no shared ancestry with HEAD; those are named and
excluded. Freeze everything at a cutoff date for reproducibility.

**Two numbers we deliberately do NOT gate on.** For releases tagged off the main line we record how
far off they sit — `branch_depth` (commits) and `divergence_days` (time). These are **diagnostic
columns, never admission gates.** A GA release is admitted because it is a GA release, not because
it is close enough to HEAD. Recording these lets a later stage reason about each snapshot's history;
it does not decide membership.

**Sequence:** configure → enumerate GA → locate vs HEAD → classify off-mainline by merge status →
admit all GA (report diagnostics, exclude only unplaceable tags) → emit grid + adapter universe →
summary.

## 1. Configuration

One frozen parameter: `GRID_CUTOFF`, a literal date (never `datetime.now()`) so a re-clone
reproduces the same grid. `TRIGGER_WINDOW_DAYS` is carried only to annotate the divergence
diagnostic; it does not gate anything here.

In [ ]:
import subprocess, re, os, json
from collections import defaultdict
from datetime import datetime, timezone
import pandas as pd

GRID_CUTOFF         = "2026-07-15"   # literal freeze date; never datetime.now()
TRIGGER_WINDOW_DAYS = 90             # annotation only in this notebook; not an admission gate

PRERELEASE_RE = re.compile(
    r'(?i)(?:^|[^a-z0-9])(rc\d*|alpha\d*|beta\d*|m\d+|milestone|snapshot|preview|dev|ea\d*|b\d+)(?:$|[^a-z0-9])')

REPO_CONFIGS = {
    "commons-lang": {"url": "https://github.com/apache/commons-lang.git", "mode": "candidate",
                     "noise": ['BETA', 'DEV', 'STRUTS', 'COPY', 'REPACKAGED', '_B']},
    "junit-framework": {"url": "https://github.com/junit-team/junit-framework.git", "mode": "allowlist",
                        "prefix": ("r5.", "r6."), "ga_regex": re.compile(r'^r(\d+)\.(\d+)\.(\d+)$')},
    "dubbo": {"url": "https://github.com/apache/dubbo.git", "mode": "allowlist",
              "ga_regex": re.compile(r'^(?:dubbo-|v)?(\d+)\.(\d+)\.(\d+)$')},
    "questdb": {"url": "https://github.com/questdb/questdb.git", "mode": "allowlist",
                "ga_regex": re.compile(r'^(?:v)?(\d+)\.(\d+)(?:\.(\d+))?$')},
    "checkstyle": {"url": "https://github.com/checkstyle/checkstyle.git", "mode": "allowlist",
                   "ga_regex": re.compile(r'^(?:checkstyle-|v|release)?(\d+)[\._](\d+)(?:[\._](\d+))?$')},
}
REPOS = list(REPO_CONFIGS)

def sh(args, cwd=None):
    return subprocess.run(args, cwd=cwd, capture_output=True, text=True)

_tcache = {}
def commit_time(sha, rd):
    k = (rd, sha)
    if k not in _tcache:
        out = sh(["git", "log", "-1", "--format=%ct", sha], cwd=rd).stdout.strip()
        _tcache[k] = int(out) if out else 0
    return _tcache[k]

def ensure_clone(name, cfg):
    if not os.path.exists(name):
        print(f"cloning {cfg['url']} ...")
        subprocess.run(["git", "clone", "--bare", cfg["url"], name], check=True)
    else:
        sh(["git", "fetch", "--tags", "--force", "origin", "+refs/heads/*:refs/heads/*"], cwd=name)
    return name

print(f"cutoff {GRID_CUTOFF}")

cutoff 2026-07-15


## 2. Enumerate GA releases

List every tag, resolve it to a commit and its true date, keep only GA releases. Pre-releases
(RC / alpha / beta / milestone) are dropped — they are not states any user ran.

Annotated tags use the *peeled* commit date (`%(*committerdate)`): the tag object's own date is
when someone typed `git tag`, not when the code was written. Four repos use one clean GA naming
scheme; commons-lang mixes three conventions, so it groups by version and lets a final RC stand in
only where a version has no GA tag at all.

In [ ]:
def enumerate_ga(name, cfg):
    rd = ensure_clone(name, cfg)
    fmt = ("%(refname:short)|%(*committerdate:iso8601)|%(committerdate:iso8601)"
           "|%(*objectname)|%(objectname)")
    out = sh(["git", "for-each-ref", f"--format={fmt}", "refs/tags"], cwd=rd).stdout
    raw = n_pre = n_nonver = 0
    ga, candidates = [], defaultdict(list)

    for line in out.strip().splitlines():
        if not line.strip():
            continue
        tag, pdate, ldate, psha, tsha = line.split('|')
        # %(*objectname) derefs ONE level. LANG_3_5 is a tag OF a tag (Apache tagged the
        # release by tagging its RC2 tag object), so one level lands on the inner tag, not
        # the commit. ^{commit} peels all the way.
        sha = sh(["git", "rev-parse", f"{tag}^{{commit}}"], cwd=rd).stdout.strip() or (psha or tsha)
        date = (pdate or ldate)[:10]
        if not date:
            date = sh(["git", "log", "-1", "--format=%cs", sha], cwd=rd).stdout.strip()
        if not date:
            continue
        raw += 1

        if cfg["mode"] == "allowlist":
            if "prefix" in cfg and not tag.startswith(cfg["prefix"]):
                n_nonver += 1; continue
            m = cfg["ga_regex"].match(tag)
            if not m:
                if re.search(r'\d+[\._]\d+', tag): n_pre += 1
                else:                                n_nonver += 1
                continue
            major, minor = int(m.group(1)), int(m.group(2))
            patch = int(m.group(3)) if (m.lastindex or 0) >= 3 and m.group(3) else 0
            ga.append(dict(tag=tag, commit_date=date, sha=sha,
                           major=major, minor=minor, patch=patch, rc_standin=False))
        else:
            if any(x in tag.upper() for x in cfg["noise"]): continue
            vm = re.search(r'(\d+)\.(\d+)(?:\.(\d+))?', tag.replace('_', '.'))
            if not vm: n_nonver += 1; continue
            key = f"{int(vm.group(1))}.{int(vm.group(2))}.{int(vm.group(3) or 0)}"
            candidates[key].append(dict(tag=tag, commit_date=date, sha=sha,
                                        major=int(vm.group(1)), minor=int(vm.group(2)),
                                        patch=int(vm.group(3) or 0),
                                        is_pre=bool(PRERELEASE_RE.search(tag))))

    if cfg["mode"] == "candidate":
        for group in candidates.values():
            finals = [g for g in group if not g["is_pre"]]
            if finals:
                pick = sorted(finals, key=lambda x: x["commit_date"])[0]; pick["rc_standin"] = False
            else:
                pick = sorted(group, key=lambda x: x["commit_date"])[-1]; pick["rc_standin"] = True
            n_pre += len(group) - 1
            pick.pop("is_pre", None)
            ga.append(pick)

    return rd, ga, dict(raw_tags=raw, non_version=n_nonver, pre_release=n_pre)

GA, REPO_DIRS = {}, {}
for name, cfg in REPO_CONFIGS.items():
    REPO_DIRS[name], GA[name], stats = enumerate_ga(name, cfg)
    print(f"{name:<17} raw {stats['raw_tags']:>4} | dropped non-version {stats['non_version']:>3}"
          f" + pre-release {stats['pre_release']:>3} | GA kept {len(GA[name]):>4}")

cloning https://github.com/apache/commons-lang.git ...
commons-lang      raw  113 | dropped non-version   0 + pre-release  67 | GA kept   36
cloning https://github.com/junit-team/junit-framework.git ...
junit-framework   raw  115 | dropped non-version   2 + pre-release  52 | GA kept   61
cloning https://github.com/apache/dubbo.git ...
dubbo             raw  158 | dropped non-version   0 + pre-release  15 | GA kept  143
cloning https://github.com/questdb/questdb.git ...
questdb           raw  131 | dropped non-version   0 + pre-release  12 | GA kept  119
cloning https://github.com/checkstyle/checkstyle.git ...
checkstyle        raw  218 | dropped non-version   2 + pre-release   5 | GA kept  211


## 3. Locate each release relative to HEAD

One yes/no per tag: *reachable from HEAD?* (`git merge-base --is-ancestor`). Reachable releases
sit on the main line. For the rest, record how far off they are — `branch_depth` (commits between
the fork point and the tag) and `divergence_days` (time between them). These are recorded for every
off-mainline release and used later only as diagnostics.

A tag with no merge-base at all (grafted pre-import SVN history) is marked **disconnected**: it
shares no ancestry with HEAD, so it cannot be placed on the grid's timeline. Sentinel `-1`.

In [ ]:
for name in REPOS:
    rd = REPO_DIRS[name]
    for g in GA[name]:
        g["version"] = f"{g['major']}.{g['minor']}.{g['patch']}"
        g["reachable"] = sh(["git", "merge-base", "--is-ancestor", g["sha"], "HEAD"],
                            cwd=rd).returncode == 0
        if g["reachable"]:
            g["branch_depth"], g["divergence_days"], g["disconnected"] = 0, 0.0, False
        else:
            mb = sh(["git", "merge-base", g["sha"], "HEAD"], cwd=rd).stdout.strip()
            if mb:
                d = sh(["git", "rev-list", "--count", f"{mb}..{g['sha']}"], cwd=rd).stdout.strip()
                g["branch_depth"] = int(d) if d else -1
                g["divergence_days"] = round(
                    (commit_time(g["sha"], rd) - commit_time(mb, rd)) / 86400.0, 1)
                g["disconnected"] = False
            else:
                g["branch_depth"], g["divergence_days"], g["disconnected"] = -1, -1.0, True
        g["after_cutoff"] = g["commit_date"] > GRID_CUTOFF

    GA[name].sort(key=lambda x: x["commit_date"])
    seen, dedup = set(), []            # SHA dedup (RC==GA double-tagging): keep earliest
    for g in GA[name]:
        if g["sha"] in seen: continue
        seen.add(g["sha"]); dedup.append(g)
    GA[name] = dedup

FRAME = pd.DataFrame([dict(repo=r, **g) for r in REPOS for g in GA[r]])
HEADS = {r: sh(["git", "rev-parse", "HEAD"], cwd=REPO_DIRS[r]).stdout.strip() for r in REPOS}

print(FRAME.groupby("repo").agg(
    ga=("tag", "count"),
    on_mainline=("reachable", "sum"),
    off_mainline=("reachable", lambda s: (~s).sum()),
    disconnected=("disconnected", "sum"),
    after_cutoff=("after_cutoff", "sum")).to_string())

                  ga  on_mainline  off_mainline  disconnected  after_cutoff
repo                                                                       
checkstyle       205          203             2             1             0
commons-lang      36           17            19             0             0
dubbo            142           62            80            19             0
junit-framework   61           17            44             0             0
questdb          119           56            63             0             0


## 4. Classify off-mainline releases by merge status

For each release **not** reachable from HEAD, determine whether its code nonetheless reached the
main line — this decides which commit the snapshot pins to, not whether it is admitted:

- **`MERGED_TRUE`** — the tag commit is an ancestor of HEAD (true / fast-forward merge).
- **`MERGED_SQUASH_OR_REBASE`** — an *identical tree* exists on a HEAD commit (squash or clean
  rebase reproduced the release under a new SHA).
- **`MERGED_PATCH_EQUIV`** — no identical tree, but `git cherry` shows every commit's patch already
  on HEAD (rebase where main shifted underneath). Tree-identity and patch-equivalence are
  complementary.
- **`UNMERGED_DANGLING`** — none of the above: the release's code lives only on its branch.

**Merged releases pin to the main-line commit** (the snapshot is effectively a HEAD state).
**Dangling releases pin to their own tag SHA** and are kept as-is — a shipped GA release is a valid
observation point whether or not it was merged forward.

In [ ]:
def classify(name, sha, tree):
    rd = REPO_DIRS[name]
    if sh(["git", "merge-base", "--is-ancestor", sha, "HEAD"], cwd=rd).returncode == 0:
        return "MERGED_TRUE", sha
    for line in sh(["git", "rev-list", "HEAD", "--format=%H %T"], cwd=rd).stdout.splitlines():
        p = line.split()
        if len(p) == 2 and p[1] == tree:
            return "MERGED_SQUASH_OR_REBASE", p[0]       # pin to the main-line twin
    cherry = [l for l in sh(["git", "cherry", "HEAD", sha], cwd=rd).stdout.splitlines() if l.strip()]
    if cherry and all(l.startswith("-") for l in cherry):
        return "MERGED_PATCH_EQUIV", None
    return "UNMERGED_DANGLING", sha

MERGED = {"MERGED_TRUE", "MERGED_SQUASH_OR_REBASE", "MERGED_PATCH_EQUIV"}

status, pin = [], []
for r in FRAME.itertuples():
    if r.reachable:
        status.append("REACHABLE"); pin.append(r.sha)
    elif r.disconnected:
        status.append("DISCONNECTED"); pin.append(None)
    else:
        tree = sh(["git", "rev-parse", f"{r.sha}^{{tree}}"], cwd=REPO_DIRS[r.repo]).stdout.strip()
        s, p = classify(r.repo, r.sha, tree)
        status.append(s); pin.append(p if p else r.sha)
FRAME["status"], FRAME["pin_sha"] = status, pin

off = FRAME[~FRAME.status.isin(["REACHABLE", "DISCONNECTED"])]
print("off-mainline releases by merge status\n")
print(off.groupby(["repo", "status"]).size().unstack(fill_value=0).to_string())
print(f"\n  merged into HEAD (pin to main line): {off.status.isin(MERGED).sum()}")
print(f"  dangling (kept as-is, own tag SHA) : {(off.status=='UNMERGED_DANGLING').sum()}")

off-mainline releases by merge status

status           MERGED_PATCH_EQUIV  MERGED_SQUASH_OR_REBASE  UNMERGED_DANGLING
repo                                                                           
checkstyle                        0                        0                  1
commons-lang                      0                        8                 11
dubbo                             3                        0                 58
junit-framework                   0                        0                 44
questdb                           0                        0                 63

  merged into HEAD (pin to main line): 11
  dangling (kept as-is, own tag SHA) : 177


## 5. Admit all GA releases and emit the grid

**Admission rule.** Every GA release is admitted, with one physical exception:

```
admit iff  commit_date <= GRID_CUTOFF
           AND NOT disconnected           # no shared ancestry with HEAD -> cannot be placed
```

No depth gate, no days gate. `branch_depth` and `divergence_days` are carried as **diagnostic
columns**. Disconnected pre-import tags are the only exclusion — they have no merge-base with HEAD,
so they cannot be located on the grid's timeline — and they are listed by name below.

Each admitted snapshot resolves to one SHA: the main-line commit for merged releases, the tag SHA
otherwise, deduplicated if a pin coincides with an existing snapshot. Output: `rel_hist_{repo}.json`
(grid) and `{repo}_release_manifest.json` (full record, including the diagnostics).

In [ ]:
DROP_TAGS = {"commons-lang": {"LANG_3_0_2_RC1"}}   # RC stand-in for a version that never shipped

def admissible(r):
    return (not r.after_cutoff) and (not r.disconnected)

FRAME["grid_admissible"] = [admissible(r) for r in FRAME.itertuples()]

RULE = (f"All GA releases (pre-releases excluded). Admit iff commit_date<={GRID_CUTOFF} AND not "
        f"disconnected-from-HEAD. No depth or days gate: branch_depth and divergence_days are "
        f"diagnostic only. Off-mainline releases merged into HEAD pin to their main-line commit; "
        f"dangling releases keep their tag SHA. Disconnected pre-import tags (no merge-base with "
        f"HEAD) are excluded and listed in the manifest.")

GRID, EXCLUDED = {}, {}
for name in REPOS:
    fr = FRAME[FRAME.repo == name]
    keep, seen = [], set()
    for g in sorted((g for g in GA[name]), key=lambda x: x["commit_date"]):
        row = fr[fr.tag == g["tag"]].iloc[0]
        if not row["grid_admissible"]:            continue
        if g["tag"] in DROP_TAGS.get(name, set()): continue
        snap = row["pin_sha"] if row["status"] in MERGED else g["sha"]
        if snap in seen:                          continue
        seen.add(snap)
        keep.append(dict(tag=g["tag"], version=g["version"], commit_date=g["commit_date"],
                         snapshot_sha=snap, tag_sha=g["sha"], status=row["status"],
                         branch_depth=int(g["branch_depth"]), divergence_days=g["divergence_days"]))
    GRID[name] = keep
    EXCLUDED[name] = dict(
        disconnected=sorted(fr[fr.disconnected]["tag"].tolist()),
        after_cutoff=sorted(fr[fr.after_cutoff]["tag"].tolist()),
        never_shipped=sorted(set(fr["tag"]) & DROP_TAGS.get(name, set())))

    json.dump(dict(repo=name, generated=datetime.now(timezone.utc).isoformat(),
                   head_sha=HEADS[name], grid_rule=RULE, cutoff=GRID_CUTOFF,
                   trigger_window_days=TRIGGER_WINDOW_DAYS, n_snapshots=len(keep),
                   excluded=EXCLUDED[name], snapshots=keep),
              open(f"{name}_release_manifest.json", "w"), indent=2)
    json.dump(dict(repo=name, grid_rule=RULE, head_sha=HEADS[name], n_snapshots=len(keep),
                   entries=[{k: s[k] for k in ("tag","version","commit_date","snapshot_sha",
                                               "status","branch_depth","divergence_days")}
                            for s in keep]),
              open(f"rel_hist_{name}.json", "w"), indent=2)

    d = [s["commit_date"] for s in keep]
    exc = EXCLUDED[name]["disconnected"]
    print(f"{name:<17} {len(keep):>4} snapshots   [{d[0]} .. {d[-1]}]"
          f"{'   excluded disconnected: ' + str(len(exc)) if exc else ''}")

TOTAL = sum(len(v) for v in GRID.values())
print(f"\nGRID TOTAL: {TOTAL} GA-release snapshots  (cutoff {GRID_CUTOFF}, no depth/days gate)")

commons-lang        35 snapshots   [2007-07-29 .. 2025-11-12]
junit-framework     61 snapshots   [2017-09-10 .. 2026-07-12]
dubbo              123 snapshots   [2012-06-26 .. 2026-05-17]   excluded disconnected: 19
questdb            119 snapshots   [2014-05-06 .. 2026-06-15]
checkstyle         204 snapshots   [2001-06-22 .. 2026-07-12]   excluded disconnected: 1

GRID TOTAL: 542 GA-release snapshots  (cutoff 2026-07-15, no depth/days gate)


### 5.1 Excluded tags, named

Every exclusion, listed so it is auditable rather than silent. `disconnected` = grafted pre-import
history with no shared ancestry with HEAD (a HEAD-based grid cannot place them). `after_cutoff` =
released after the freeze date. `never_shipped` = an RC stand-in for a version that was never
actually released.

In [ ]:
for name in REPOS:
    e = EXCLUDED[name]
    total = len(e["disconnected"]) + len(e["after_cutoff"]) + len(e["never_shipped"])
    if not total:
        print(f"{name:<17} nothing excluded"); continue
    print(f"{name}:")
    if e["disconnected"]:
        print(f"   disconnected ({len(e['disconnected'])}): "
              f"{e['disconnected'][0]} .. {e['disconnected'][-1]}")
    if e["after_cutoff"]:
        print(f"   after cutoff ({len(e['after_cutoff'])}): {', '.join(e['after_cutoff'])}")
    if e["never_shipped"]:
        print(f"   never shipped: {', '.join(e['never_shipped'])}")

commons-lang:
   never shipped: LANG_3_0_2_RC1
junit-framework   nothing excluded
dubbo:
   disconnected (19): dubbo-2.0.10 .. dubbo-2.3.0
questdb           nothing excluded
checkstyle:
   disconnected (1): release3_5 .. release3_5


## 6. The commit universe for the mining adapters

The grid records, per repo, the snapshots whose code is **not** on HEAD (the dangling releases) —
their branch commits must be walked explicitly so their history is available downstream. This
writes `adapter_universe_{repo}.json`: HEAD plus the dangling snapshot SHAs, never `--all`.

Reachable and merged snapshots contribute nothing here — their code is already on HEAD. This file
records *which* commits belong to the study universe beyond HEAD; it does not prescribe how any
adapter uses them.

In [ ]:
for name in REPOS:
    rd = REPO_DIRS[name]
    dangling = [s["snapshot_sha"] for s in GRID[name] if s["status"] == "UNMERGED_DANGLING"]
    n_head = int(sh(["git", "rev-list", "--count", "HEAD"], cwd=rd).stdout.strip())
    n_all  = int(sh(["git", "rev-list", "--count", "HEAD", *dangling], cwd=rd).stdout.strip()) \
             if dangling else n_head
    json.dump(dict(repo=name, head_sha=HEADS[name], head_commits=n_head,
                   near_mainline_shas=dangling, rev_list_args=["HEAD"] + dangling,
                   universe_commits=n_all, extra_commits=n_all - n_head),
              open(f"adapter_universe_{name}.json", "w"), indent=2)
    print(f"{name:<17} HEAD {n_head:>6} + {len(dangling):>3} dangling = {n_all:>6} "
          f"(+{n_all - n_head} commits)")

commons-lang      HEAD   9641 +  11 dangling =   9883 (+242 commits)
junit-framework   HEAD  10925 +  44 dangling =  12172 (+1247 commits)
dubbo             HEAD   8891 +  58 dangling =   9956 (+1065 commits)
questdb           HEAD   6229 +  63 dangling =   6340 (+111 commits)
checkstyle        HEAD  17181 +   1 dangling =  17185 (+4 commits)


## 7. Depth & days diagnostic (reported, not gated)

Neither number gates admission. Their distribution across the dangling releases is reported as a
description of the grid — how far off-mainline the kept releases sit — not as a filter. `branch_depth`
is the branch length; `divergence_days` is annotated against the study's 90-day trigger window purely
for context.

In [ ]:
dang = FRAME[(FRAME.status == "UNMERGED_DANGLING") & FRAME.grid_admissible]
print(f"admitted dangling releases: {len(dang)}\n")
if len(dang):
    print(dang.groupby("repo").agg(
        n=("tag", "count"),
        depth_min=("branch_depth", "min"), depth_max=("branch_depth", "max"),
        depth_median=("branch_depth", "median"),
        days_min=("divergence_days", "min"), days_max=("divergence_days", "max"),
        within_90d=("divergence_days", lambda s: int((s <= TRIGGER_WINDOW_DAYS).sum())),
    ).to_string())
    print(f"\nreported only. depth spans {int(dang.branch_depth.min())}-"
          f"{int(dang.branch_depth.max())} commits; "
          f"{int((dang.divergence_days <= TRIGGER_WINDOW_DAYS).sum())}/{len(dang)} "
          f"within the {TRIGGER_WINDOW_DAYS}-day window.")

admitted dangling releases: 177

                  n  depth_min  depth_max  depth_median  days_min  days_max  within_90d
repo                                                                                   
checkstyle        1          4          4           4.0      34.8      34.8           1
commons-lang     11          2        192           3.0       3.2    1761.3           6
dubbo            58          1        654          53.0       0.1    1295.2          28
junit-framework  44          1        319          53.5       4.0     439.0          25
questdb          63          1         40           1.0      -0.0     249.8          61

reported only. depth spans 1-654 commits; 121/177 within the 90-day window.


## 8. Summary

The grid is every GA release in each project's continuous (HEAD-connected) history, mined by one
reproducible rule:

1. **GA releases only** — pre-releases excluded (§2).
2. **Reachable releases** used as-is (§3).
3. **Off-mainline releases classified by merge status** (§4): merged ones pin to their main-line
   commit; dangling ones keep their tag SHA.
4. **All GA releases admitted** (§5) — no depth or days gate. A shipped release is a valid
   observation point regardless of branch. `branch_depth` and `divergence_days` are recorded as
   diagnostics (§7).
5. **Only unplaceable tags excluded** — disconnected pre-import history with no shared ancestry
   with HEAD — listed by name (§5.1).
6. **Frozen** at the cutoff for reproducibility.

Every admitted snapshot resolves to a single commit SHA with a recorded merge status and
diagnostics, so the grid, the manifests, and the adapter commit universe are consistent by
construction.